<a href="https://colab.research.google.com/github/Anisha2810/Neural-networks-and-deep-learning/blob/main/IMPLEMENTATION_OF_GENERATIVE_ADVERSARIAL_NETWORK.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# 1️⃣ Install required packages
!pip -q install torch torchvision matplotlib

import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torchvision.utils import save_image
import matplotlib.pyplot as plt
import os

# 2️⃣ Hyperparameters
batch_size = 64
z_dim = 100   # noise vector size
lr = 0.0002
num_epochs = 10
image_size = 28*28  # MNIST images flattened
device = 'cuda' if torch.cuda.is_available() else 'cpu'

# 3️⃣ Load MNIST dataset
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

train_loader = torch.utils.data.DataLoader(
    datasets.MNIST('.', train=True, download=True, transform=transform),
    batch_size=batch_size, shuffle=True
)

# 4️⃣ Define Generator
class Generator(nn.Module):
    def __init__(self, z_dim, img_dim):
        super(Generator, self).__init__()
        self.model = nn.Sequential(
            nn.Linear(z_dim, 256),
            nn.ReLU(True),
            nn.Linear(256, 512),
            nn.ReLU(True),
            nn.Linear(512, img_dim),
            nn.Tanh()
        )
    def forward(self, z):
        return self.model(z)

# 5️⃣ Define Discriminator
class Discriminator(nn.Module):
    def __init__(self, img_dim):
        super(Discriminator, self).__init__()
        self.model = nn.Sequential(
            nn.Linear(img_dim, 512),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Linear(512, 256),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Linear(256, 1),
            nn.Sigmoid()
        )
    def forward(self, x):
        return self.model(x)

# 6️⃣ Initialize networks
G = Generator(z_dim, image_size).to(device)
D = Discriminator(image_size).to(device)

# 7️⃣ Loss and optimizers
criterion = nn.BCELoss()
optimizer_G = optim.Adam(G.parameters(), lr=lr)
optimizer_D = optim.Adam(D.parameters(), lr=lr)

# 8️⃣ Training loop
for epoch in range(num_epochs):
    for i, (imgs, _) in enumerate(train_loader):
        imgs = imgs.view(-1, image_size).to(device)
        batch_size_curr = imgs.size(0)

        # Labels
        real_labels = torch.ones(batch_size_curr, 1).to(device)
        fake_labels = torch.zeros(batch_size_curr, 1).to(device)

        # ---------------------
        # Train Discriminator
        # ---------------------
        z = torch.randn(batch_size_curr, z_dim).to(device)
        fake_imgs = G(z)

        D_real = D(imgs)
        D_fake = D(fake_imgs.detach())

        loss_D = criterion(D_real, real_labels) + criterion(D_fake, fake_labels)
        optimizer_D.zero_grad()
        loss_D.backward()
        optimizer_D.step()

        # ---------------------
        # Train Generator
        # ---------------------
        D_fake = D(fake_imgs)
        loss_G = criterion(D_fake, real_labels)
        optimizer_G.zero_grad()
        loss_G.backward()
        optimizer_G.step()

    print(f"Epoch [{epoch+1}/{num_epochs}] | Loss D: {loss_D.item():.4f} | Loss G: {loss_G.item():.4f}")

    # Save generated images
    if (epoch+1) % 2 == 0:
        save_image(fake_imgs.view(-1, 1, 28, 28)[:25],
                   f"gan_epoch_{epoch+1}.png", nrow=5, normalize=True)

# 9️⃣ Display last generated images
fake_imgs = fake_imgs.view(-1, 1, 28, 28)
grid_img = torchvision.utils.make_grid(fake_imgs[:25], nrow=5, normalize=True)
plt.figure(figsize=(6,6))
plt.imshow(grid_img.permute(1, 2, 0).cpu())
plt.axis('off')
plt.show()

100%|██████████| 9.91M/9.91M [00:00<00:00, 41.6MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 1.14MB/s]
100%|██████████| 1.65M/1.65M [00:00<00:00, 10.5MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 6.20MB/s]


Epoch [1/10] | Loss D: 0.9381 | Loss G: 6.2478
Epoch [2/10] | Loss D: 0.2761 | Loss G: 3.6385
